# Smart Contract Vulnerability Detection - GPT-2 Fine-tuning
Training with all optimization thresholds: before_optimized, optimized_80p, optimized_50p, optimized_20p

Each model is trained from scratch with fresh parameters loaded from `MODEL_NAME`.

---
**Edit the constants in the cell below to configure the run.**

In [ ]:
# Run and save all Notebook
# SSH into tmux
# uv run jupyter nbconvert --to notebook --execute --inplace --allow-errors /workspace/smart-contract-vulnerability-detection-gpt-2.ipynb
# Ctrl + B, then D to detach
# After run notebook, ensure to check notebook, download, change dataset name, and run new session
# Stop instance to do later but must check for disk price

# ============================================================
# CELL 0 — GLOBAL CONSTANTS (edit here to configure the run)
# ============================================================

# HuggingFace model hub ID used for tokeniser + base weights
MODEL_NAME = "gpt2"

# HuggingFace dataset hub ID  (must expose train / test splits)
DATASET_NAME = "JakeClark/soliaudit-dasp-sequence-gnn-explainer"

# Text columns to iterate over during training
TEXT_COLUMNS = ["before_optimized", "optimized_80p", "optimized_50p", "optimized_20p"]

# Multi-label target columns
LABEL_COLUMNS = [
    "Arithmetic",
    "Unchecked Return Values For Low Level Calls",
    "Denial of Service",
    "Time manipulation",
    "Reentrancy",
]

# ---------- Training hyper-parameters ----------
NUM_EPOCHS          = 10
TRAIN_BATCH_SIZE    = 8
EVAL_BATCH_SIZE     = 8
LEARNING_RATE       = 2e-5
WEIGHT_DECAY        = 0.01
MAX_TOKEN_LENGTH    = 1024

# ---------- Output ----------
# Where Trainer saves checkpoints (and where we look for them to resume)
OUTPUT_BASE = "/working/data"

# ---------- HuggingFace auth (optional — set if dataset/model is private) ----------
# Leave as None to use the environment variable HF_TOKEN automatically.
HF_TOKEN = None   # e.g. "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

print("Global constants loaded.")
print(f"  MODEL_NAME   : {MODEL_NAME}")
print(f"  DATASET_NAME : {DATASET_NAME}")
print(f"  TEXT_COLUMNS : {TEXT_COLUMNS}")
print(f"  OUTPUT_BASE  : {OUTPUT_BASE}")

In [ ]:
import os, sys, time, subprocess
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print()

In [ ]:
print("="*60)
print("STEP 1: Install Dependencies")
print("="*60)

!source /venv/main/bin/activate
!/venv/main/bin/python -m pip install pandas transformers scikit-learn accelerate datasets

for pkg in ["transformers", "datasets", "pandas", "sklearn"]:
    try:
        mod = __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
print()

In [ ]:
print("="*60)
print("STEP 2: Load Dataset from HuggingFace Hub")
print("="*60)

import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

# Authenticate if a token was provided in the constants cell
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to HuggingFace Hub.")
else:
    print("No HF_TOKEN set — using public access / env HF_TOKEN.")

print(f"\nLoading dataset: {DATASET_NAME}")
hf_dataset = load_dataset(DATASET_NAME)

# Convert to pandas DataFrames
train_df = hf_dataset["train"].to_pandas()
test_df  = hf_dataset["test"].to_pandas()

print(f"Train samples : {len(train_df)}")
print(f"Test samples  : {len(test_df)}")
print(f"Columns       : {train_df.columns.tolist()}")
print(f"Labels        : {LABEL_COLUMNS}")
print()

In [ ]:
print("="*60)
print("STEP 3: Define Helper Functions")
print("="*60)

import os
import gc
import numpy as np
from pathlib import Path
from sklearn.metrics import classification_report, hamming_loss, precision_recall_fscore_support
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer, Trainer, TrainingArguments
from torch.utils.data import Dataset as TorchDataset
from torch.utils.data import DataLoader
import torch

def format_vulnerabilities(labels, label_columns):
    vulns = []
    for i, label in enumerate(label_columns):
        if labels[i] == 1:
            vulns.append(label)
    if not vulns:
        vulns.append("None")
    return "Vulnerabilities:\n" + "\n".join(f"* {v}" for v in vulns)

def parse_vulnerabilities(output_text, label_columns):
    output_lower = output_text.lower()
    detected = []
    for label in label_columns:
        label_lower = label.lower()
        if label_lower in output_lower:
            detected.append(1)
        else:
            detected.append(0)
    if sum(detected) == 0:
        detected = [0] * len(label_columns)
    return detected

class VulnerabilityClassificationDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_length=1024):
        self.texts = texts
        # Labels must be float32 for multi-label BCE loss
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.texts)

def hamming_score(y_true, y_pred, normalize=True, sample_weight=None):
    acc_list = []
    for i in range(y_true.shape[0]):
        set_true = set(np.where(y_true[i])[0])
        set_pred = set(np.where(y_pred[i])[0])
        tmp_a = None
        if len(set_true) == 0 and len(set_pred) == 0:
            tmp_a = 1
        else:
            tmp_a = len(set_true.intersection(set_pred))/float(len(set_true.union(set_pred)))
        acc_list.append(tmp_a)
    return np.mean(acc_list)

def _latest_checkpoint(directory: str):
    """Return the path of the most-recent Trainer checkpoint in *directory*,
    or None if no checkpoint exists."""
    ckpt_dir = Path(directory)
    if not ckpt_dir.exists():
        return None
    checkpoints = sorted(
        [p for p in ckpt_dir.iterdir() if p.name.startswith("checkpoint-")],
        key=lambda p: int(p.name.split("-")[-1]),
    )
    return str(checkpoints[-1]) if checkpoints else None

def evaluate_model(model, test_dataset, batch_size=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    all_preds = []
    all_labels = []

    print(f"Starting inference on {len(test_dataset)} samples...")

    with torch.no_grad():
        for batch in test_loader:
            # Move batch to GPU
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Convert logits to probabilities using Sigmoid
            # Formula: 1 / (1 + exp(-x))
            probs = torch.sigmoid(logits)

            # Threshold at 0.5 to get binary predictions
            preds = (probs > 0.5).int()

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    # Flatten results
    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    return y_true, y_pred

def train_and_evaluate(column_name, train_df, test_df, output_dir):
    print(f"\n" + "="*60)
    print(f"Training with column: {column_name}")
    print("="*60)

    col_output_dir = os.path.join(output_dir, column_name)
    Path(col_output_dir).mkdir(parents=True, exist_ok=True)

    # ── Check for an existing checkpoint to resume from ──────────────
    resume_from = _latest_checkpoint(col_output_dir)
    if resume_from:
        print(f"[RESUME] Found checkpoint: {resume_from}")
    else:
        print("[START] No checkpoint found — training from scratch.")

    train_texts = train_df[column_name].fillna("").astype(str).tolist()
    test_texts = test_df[column_name].fillna("").astype(str).tolist()
    train_labels = train_df[LABEL_COLUMNS].values
    test_labels = test_df[LABEL_COLUMNS].values

    print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

    tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left" # CRITICAL for GPT-2 classification

    train_dataset = VulnerabilityClassificationDataset(train_texts, train_labels, tokenizer, max_length=MAX_TOKEN_LENGTH)
    test_dataset = VulnerabilityClassificationDataset(test_texts, test_labels, tokenizer, max_length=MAX_TOKEN_LENGTH)

    model = GPT2ForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLUMNS),
        problem_type="multi_label_classification" # This automatically sets the right loss
    )
    model.config.pad_token_id = tokenizer.eos_token_id

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    train_size = int(0.8 * len(train_dataset))
    eval_size = len(train_dataset) - train_size
    train_dataset_split, eval_dataset = torch.utils.data.random_split(
        train_dataset, [train_size, eval_size]
    )

    training_args = TrainingArguments(
        output_dir=col_output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=False,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        logging_steps=100,
        report_to="none",
        gradient_accumulation_steps=2,
        eval_accumulation_steps=10,  # Move to CPU every 10 batches
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset_split,
        eval_dataset=eval_dataset,
    )

    train_start = time.time()
    print("Training...")
    trainer.train(resume_from_checkpoint=resume_from)
    train_time = time.time() - train_start

    print("Generating predictions...")
    test_start = time.time()
    test_labels, pred_labels = evaluate_model(model, test_dataset, batch_size=EVAL_BATCH_SIZE)
    test_inference_time = time.time() - test_start

    print("\nClassification Report by Label:")
    print(classification_report(test_labels, pred_labels, target_names=LABEL_COLUMNS, zero_division=0))

    precision, recall, f1, _ = precision_recall_fscore_support(
        test_labels, pred_labels, average="weighted", zero_division=0
    )
    hamming = hamming_score(test_labels, pred_labels)
    h_loss = hamming_loss(test_labels, pred_labels)

    model.save_pretrained(col_output_dir)
    tokenizer.save_pretrained(col_output_dir)
    print(f"Model saved to: {col_output_dir}")

    del model, trainer, train_dataset, test_dataset
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "column": column_name,
        "train_time": train_time,
        "train_inference_time": 0,
        "test_inference_time": test_inference_time,
        "num_train_samples": len(train_texts),
        "num_test_samples": len(test_texts),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "hamming_score": hamming,
        "hamming_loss": h_loss,
    }

print("Helper functions defined")
print()

In [ ]:
print("="*60)
print("STEP 4: Run All Experiments")
print("="*60)

import json
from pathlib import Path

Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)

all_results = []
total_start = time.time()

for col in TEXT_COLUMNS:
    result = train_and_evaluate(
        column_name=col,
        train_df=train_df,
        test_df=test_df,
        output_dir=OUTPUT_BASE
    )
    all_results.append(result)

    print(f"\nResults for {col}:")
    print(f"  Train Samples: {result['num_train_samples']}, Test Samples: {result['num_test_samples']}")
    print(f"  Train Time: {result['train_time']/60:.1f} min")
    print(f"  Test Inference Time: {result['test_inference_time']:.2f}s")
    print(f"  Precision: {result['precision']:.4f}")
    print(f"  Recall: {result['recall']:.4f}")
    print(f"  F1: {result['f1']:.4f}")
    print(f"  Hamming Score: {result['hamming_score']:.4f}")
    print(f"  Hamming Loss: {result['hamming_loss']:.4f}")

    gc.collect()
    torch.cuda.empty_cache()

total_time = time.time() - total_start
print(f"\nTotal training time: {total_time/60:.1f} minutes")
print()

In [ ]:
print("="*60)
print("STEP 5: Summary Comparison")
print("="*60)

import json
import pandas as pd

comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[["column", "num_train_samples", "num_test_samples", "train_time", "train_inference_time", "test_inference_time", "precision", "recall", "f1", "hamming_score", "hamming_loss"]]
comparison_df.columns = ["Dataset", "Train Samples", "Test Samples", "Train Time (s)", "Train Inference (s)", "Test Inference (s)", "Precision", "Recall", "F1", "Hamming Score", "Hamming Loss"]

print("\n" + "="*100)
print("FINAL RESULTS COMPARISON")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

comparison_csv = os.path.join(OUTPUT_BASE, "comparison_results.csv")
comparison_df.to_csv(comparison_csv, index=False)
print(f"\nResults saved to: {comparison_csv}")

results_json = {
    "configuration": {
        "model":        MODEL_NAME,
        "dataset":      DATASET_NAME,
        "text_columns": TEXT_COLUMNS,
        "labels":       LABEL_COLUMNS,
        "num_epochs":   NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
    },
    "results": all_results,
    "total_time": total_time
}

results_json_path = os.path.join(OUTPUT_BASE, "experiment_results.json")
with open(results_json_path, "w") as f:
    json.dump(results_json, f, indent=2)
print(f"Results saved to: {results_json_path}")

print("\nAll experiments completed!")
print()